In [1]:
import pandas as pd
import numpy as np
from pandas import DataFrame, read_excel
from sklearn.preprocessing import MinMaxScaler
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import accuracy_score, recall_score, make_scorer
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.svm import SVC
from sklearn.neural_network import MLPClassifier

In [ ]:
filename = "895_records_with_descriptors_with_names.xlsx"
dataset: DataFrame = read_excel(filename)
print(dataset.shape)

##### **Variables Enconding**

In [ ]:
dissolve_encoding: dict[str, int] = {"NO": 0,"YES": 1}
sample_type_enconding: dict[str, int] = {"pellet": 0, "waste": 1, "fiber": 2, "film": 3, "powder": 4}

encoding: dict[str, dict[str, int]] = {
    "Dissolution": dissolve_encoding,
    "Sample type": sample_type_enconding
}
df: DataFrame = dataset.replace(encoding, inplace=False)

##### **Variables Normalization**

In [4]:
# Remove Polymer and Solvent Identifier Columns
df = df.drop(columns=["Polymer_ID", "Solvent_ID", "Polymer", "Solvent"])

# Separate features (X) from target "Dissolution" (y)
X = df.drop(columns=["Dissolution"])
y = df["Dissolution"]

# Apply log(1+x) Transformation
X_log = np.log1p(X)

# Scale to [0,1] with MinMax Normalization
min_max_scaler = MinMaxScaler(feature_range=(0, 1), copy=True)
X_scaled = min_max_scaler.fit_transform(X_log)

# Rebuild Dataset merging normalized features with target
df_log_minmax = DataFrame(X_scaled, columns = X.columns, index= X.index)
df_log_minmax["Dissolution"] = y

Cross-validation

In [5]:
# Parameters
target = 'Dissolution'
X = df_log_minmax.drop(columns=target)
y = df_log_minmax[target]

DT

In [ ]:
# Hyperparameters to test
param_grid = {
    'max_depth': [2, 3, 4, 5, 6, 7, 8, 9, 10],
    'min_samples_split': [2, 3, 4, 5],
    'min_samples_leaf': [1, 5, 10],
    'max_features': [None, 'sqrt', 'log2', 0.5, 0.75],
    'criterion': ['entropy', 'gini'],
    'splitter': ['best'],
    'min_impurity_decrease': [0.0, 0.0001, 0.001, 0.01]
}

# save results
acc_val_list = []
acc_train_list = []
acc_test_list = []
recall_test_list = []
best_params_list = []

# Loop of iterations with random_state varying
for i in range(5):  # number of iterations
    # Dataset division
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=i, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=i, stratify=y_temp)

    print(f"\n{'='*30}\nStarting Iteration {i}\n{'='*30}")
 
    # GridSearch in validation
    grid_search = GridSearchCV(DecisionTreeClassifier(random_state=i), param_grid, cv=5, scoring='accuracy')
    grid_search.fit(X_val, y_val)
    best_params = grid_search.best_params_
    best_params_list.append(best_params)

    # Train model with better hyperparameters
    model = DecisionTreeClassifier(**best_params, random_state=i)
    model.fit(X_train, y_train)

    # Predictions
    y_val_pred = model.predict(X_val)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Calculate and store metrics
    acc_val = accuracy_score(y_val, y_val_pred)
    acc_train = accuracy_score(y_train, y_train_pred)
    acc_test = accuracy_score(y_test, y_test_pred)
    recall_test = recall_score(y_test, y_test_pred)

    acc_val_list.append(acc_val)
    acc_train_list.append(acc_train)
    acc_test_list.append(acc_test)
    recall_test_list.append(recall_test)

    # Show iteration results
    print(f"Best otimization:  {best_params}")
    print(f"Accuracy - Validation:  {acc_val:.3f}")
    print(f"Accuracy - Train:     {acc_train:.3f}")
    print(f"Accuracy - Test:      {acc_test:.3f}")



print("\n=== Final Results ===")
print(f"Accuracy - Validation: Mean = {np.mean(acc_val_list):.3f}, Standard Deviation = {np.std(acc_val_list):.3f}")
print(f"Accuracy - Train:     Mean = {np.mean(acc_train_list):.3f}, Standard Deviation = {np.std(acc_train_list):.3f}")
print(f"Accuracy - Test:      Mean = {np.mean(acc_test_list):.3f}, Standard Deviation = {np.std(acc_test_list):.3f}")

GB

In [ ]:
# Hyperparameters to test

param_grid = {
    'n_estimators': [10, 20, 30],
    'learning_rate': [0.01, 0.04, 0.1, 0.2, 0.3],
    'max_depth': [5, 7, 9],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 5, 10],
    'subsample': [0.8, 1.0],
    'max_features': ['sqrt', None, 'log2'],
    'loss': ['log_loss', 'exponential']
}


# save results
acc_val_list = []
acc_train_list = []
acc_test_list = []
recall_test_list = []
best_params_list = []

# Loop of iterations with random_state varying
for i in range(5):  # number of iterations
    # Dataset division
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=i, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=i, stratify=y_temp)

    print(f"\n{'='*30}\nStarting Iteration {i}\n{'='*30}")
 

    # GridSearch in validation
    grid_search = GridSearchCV(GradientBoostingClassifier(random_state=i), param_grid, cv=5, scoring='accuracy', n_jobs =-1)
    grid_search.fit(X_val, y_val)
    best_params = grid_search.best_params_
    best_params_list.append(best_params)

    # Train model with better hyperparameters
    model = GradientBoostingClassifier(**best_params, random_state=i)
    model.fit(X_train, y_train)

    # Predictions
    y_val_pred = model.predict(X_val)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Calculate and store metrics
    acc_val = accuracy_score(y_val, y_val_pred)
    acc_train = accuracy_score(y_train, y_train_pred)
    acc_test = accuracy_score(y_test, y_test_pred)
    recall_test = recall_score(y_test, y_test_pred)

    acc_val_list.append(acc_val)
    acc_train_list.append(acc_train)
    acc_test_list.append(acc_test)
    recall_test_list.append(recall_test)

    # Show iteration results
    print(f"Best otimization:  {best_params}")
    print(f"Accuracy - Validation:  {acc_val:.3f}")
    print(f"Accuracy - Train:     {acc_train:.3f}")
    print(f"Accuracy - Test:      {acc_test:.3f}")


print("\n=== Final Results ===")
print(f"Accuracy - Validation: Mean = {np.mean(acc_val_list):.3f}, Standard Deviation = {np.std(acc_val_list):.3f}")
print(f"Accuracy - Train:     Mean = {np.mean(acc_train_list):.3f}, Standard Deviation = {np.std(acc_train_list):.3f}")
print(f"Accuracy - Test:      Mean = {np.mean(acc_test_list):.3f}, Standard Deviation = {np.std(acc_test_list):.3f}")


MLP

In [ ]:
# Hyperparameters to test

param_grid = {
    'hidden_layer_sizes': [(40,), (60,), (40, 80), (60,80), (60,60), (80,40)],
    'activation': ['tanh', 'relu', 'logistic'],
    'solver': ['adam', 'sgd'],
    'learning_rate': ['constant', 'adaptive'],
    'alpha': [0.0001, 0.001, 0.005, 0.01, 0.02],
    'learning_rate_init': [0.5, 0.05],
    'max_iter': [200, 500, 1000, 2500, 3000]
}


# save results
acc_val_list = []
acc_train_list = []
acc_test_list = []
recall_test_list = []
best_params_list = []

# Loop of iterations with random_state varying
for i in range(5):  # number of iterations
    # Dataset division
    X_train, X_temp, y_train, y_temp = train_test_split(X, y, test_size=0.30, random_state=i, stratify=y)
    X_val, X_test, y_val, y_test = train_test_split(X_temp, y_temp, test_size=0.5, random_state=i, stratify=y_temp)

    print(f"\n{'='*30}\nStarting Iteration {i}\n{'='*30}")
 

    # GridSearch in validation
    grid_search = GridSearchCV(MLPClassifier(random_state=i), param_grid, cv=5, scoring='accuracy')
    grid_search.fit(X_val, y_val)
    best_params = grid_search.best_params_
    best_params_list.append(best_params)

    # Train model with better hyperparameters
    model = MLPClassifier(**best_params, random_state=i)
    model.fit(X_train, y_train)

    # Predictions
    y_val_pred = model.predict(X_val)
    y_train_pred = model.predict(X_train)
    y_test_pred = model.predict(X_test)

    # Calculate and store metrics
    acc_val = accuracy_score(y_val, y_val_pred)
    acc_train = accuracy_score(y_train, y_train_pred)
    acc_test = accuracy_score(y_test, y_test_pred)
    recall_test = recall_score(y_test, y_test_pred)

    acc_val_list.append(acc_val)
    acc_train_list.append(acc_train)
    acc_test_list.append(acc_test)
    recall_test_list.append(recall_test)

    if i == 0 or acc_test > max(acc_test_list[:-1]):
        best_model = model
        best_params_final = best_params
        best_X = pd.concat([X_train, X_val, X_test])

    # Show iteration results
    print(f"Best otimization:  {best_params}")
    print(f"Accuracy - Validation:  {acc_val:.3f}")
    print(f"Accuracy - Train:     {acc_train:.3f}")
    print(f"Accuracy - Test:      {acc_test:.3f}")


# Cálculo de médias e desvios padrão após todas as iterações
print("\n=== Final Results ===")
print(f"Accuracy - Validation: Mean = {np.mean(acc_val_list):.3f}, Standard Deviation = {np.std(acc_val_list):.3f}")
print(f"Accuracy - Train:     Mean = {np.mean(acc_train_list):.3f}, Standard Deviation = {np.std(acc_train_list):.3f}")
print(f"Accuracy - Test:      Mean = {np.mean(acc_test_list):.3f}, Standard Deviation = {np.std(acc_test_list):.3f}")